## MODULE 4 — Data Cleaning
---
**Real-world data is messy.** This module covers every technique you will use in production.

In [100]:
import numpy as np
import pandas as pd

np.random.seed(7)
n = 150

messy = pd.DataFrame({
    'Customer ID': list(range(1, n+1)) + [5, 23, 67],

    'customer_name': (['Alice Johnson', 'BOB smith', '  carol white  ', 
                       'David Brown', 'Eve Davis'] * 30) + ['Frank Miller'] * 3,

    'Age': np.random.choice(list(range(20, 70)) + [None, -5, 999], n+3),

    'email': (['user@example.com', 'INVALID_EMAIL', None, 
               'test@test.com', 'jane@corp.com'] * 30) + ['a@b.com'] * 3,

    'revenue': list(np.random.uniform(100, 10000, n)) + [None, None, None],

    'signup_date': (['2024-01-15', '15/02/2024', 'March 3 2024', 
                     '2024-04-01', 'not a date'] * 30) + ['2024-05-01'] * 3,

    'country': (['USA', 'US', 'United States', 'uk', 'UK', 
                 'United Kingdom', 'CA', 'Canada'] * 18) + ['USA'] * 9
})

messy = messy.sample(frac=1, random_state=42).reset_index(drop=True)

print(messy.shape)

(153, 7)


In [103]:
# 1. Data Audit 
df = messy.copy() # always work on a copy of the original data

df.info()
df.describe()
df.isnull().sum()



<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer ID    153 non-null    int64  
 1   customer_name  153 non-null    str    
 2   Age            151 non-null    object 
 3   email          123 non-null    str    
 4   revenue        150 non-null    float64
 5   signup_date    153 non-null    str    
 6   country        153 non-null    str    
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 8.5+ KB


Customer ID       0
customer_name     0
Age               2
email            30
revenue           3
signup_date       0
country           0
dtype: int64

In [104]:
#2. Handle Missing Values (IMPORTANT)
df.dropna(how='all', inplace=True)

df['revenue'] = df['revenue'].fillna(df['revenue'].median())
df['email'] = df['email'].fillna('unknown')


In [105]:
# 3. Remove Duplicates
df = df.sort_values('revenue', ascending=False)\
       .drop_duplicates(subset=['Customer ID'], keep='first')



In [ ]:

# 4. Fix Data Types
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# ADD THIS HERE
df.loc[~df['Age'].between(18, 100), 'Age'] = np.nan
df['Age'] = df['Age'].fillna(df['Age'].median())

median_date = df['signup_date'].dropna().median()
df['signup_date'] = df['signup_date'].fillna(median_date)

In [107]:
# 5. Handle Outliers (ONLY revenue, NOT Age again)
df['revenue'] = df['revenue'].clip(upper=100000)


In [114]:
# 6. Standardization

df['country'] = df['country'].replace({
    'US': 'USA',
    'United States': 'USA',
    'uk': 'UK'
})
#print(df.isnull().sum())
#print(df.duplicated().sum())
#print(df.shape)
print(df.head())

     Customer ID    customer_name Age          email      revenue  \
41            65        Eve Davis  53  jane@corp.com  9926.321420   
145          122        BOB smith  64  INVALID_EMAIL  9887.479912   
90            35        Eve Davis  58  jane@corp.com  9753.882843   
142          117        BOB smith  49  INVALID_EMAIL  9737.408126   
111          118    carol white    28        unknown  9558.296173   

      signup_date         country  
41     not a date             USA  
145    15/02/2024             USA  
90     not a date             USA  
142    15/02/2024              UK  
111  March 3 2024  United Kingdom  


#### Data Cleaning flow = Copy → Missing → Duplicates → Types+Validation → Outliers → Standardization → Check